# Quick tests

In [ ]:
import pandas as pd
import tomllib
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Any
from pathlib import Path


## attempt for interpolation

In [ ]:
# load example data
infile = Path(Path.cwd()).resolve().parents[1] / "data" / "clean" / "fulldata.parquet"
pd.read_parquet(infile)  # .pivot(index=['census_year','vintage'], columns='type')
pd.read_parquet(infile)["vintage"].unique()
df = pd.read_parquet(infile)
# TEST to apply mask
# mask = df['vintage'].apply(lambda vintage: int( vintage.split('-')[0]) > 1990)
# df.loc[mask, 'dwellings'] = 9876789
# df['vintage'][0]

In [ ]:
# Look for nans
test = df.pivot(index=["census_year", "vintage"], columns="type")
test[test.isna().any(axis=1)]  # There are 109 rows with nans

In [ ]:
# Test for typesplit based on future interpolation
# This shows the typesplit before interpolation
tsplit = pd.DataFrame(
    df.groupby(by=["census_year", "type"])["dwellings"].max()
    / df.groupby(by=["census_year"])["dwellings"].first()
).reset_index()

g = sns.relplot(
    data=tsplit[
        tsplit["type"].isin(
            ["total", "single_detached", "single_attached", "apartments", "mobile"]
        )
    ],
    x="census_year",
    y="dwellings",
    hue="type",
    kind="line",
)
g.set(xlim=(1900, 2000))

In [ ]:
# new approach for tsplit
# Get number of dwellings per type, (total, i.e., 1608-2025) 
s = df[df['vintage']=='1608-2025'].groupby(by=["census_year", "type"])["dwellings"].sum()

shares = (s / s.xs('total', level='type')).reset_index()


g = sns.relplot(
    data=shares[shares['type'].isin(["total", "single_detached", "single_attached", "apartments", "mobile"])],
    x="census_year",
    y="dwellings",
    hue="type",
    kind="line",
)
g.set(xlim=(1900, 2000))

In [ ]:
# Interpolation method #1
df2 = []
groups = df.groupby(by=["vintage", "type"])
for name, group in groups:
    group = group.bfill()
    df2.append(group)

df2 = pd.concat(df2)

tsplit2 = pd.DataFrame(
    df2.groupby(by=["census_year", "type"])["dwellings"].max()
    / df2.groupby(by=["census_year"])["dwellings"].first()
).reset_index()

g = sns.relplot(
    data=tsplit2[
        tsplit2["type"].isin(
            ["total", "single_detached", "single_attached", "apartments", "mobile"]
        )
    ],
    x="census_year",
    y="dwellings",
    hue="type",
    kind="line",
)
g.set(xlim=(1900, 2000))

of course, this doesn't work - we would at least need to do a IPFN pass to adjust weights before calculating typesplit, and even then, the weights would be wrong, since nothing guarantees that the data for different (types, cohorts) would come from the same year. So a 1981 apartment count could be pulled, and a 1921 mobile count, and then the IPFN would adjust weights based on this data. This would of course introduce wrong weights to older data.

In [ ]:
# Test new approach
df.pivot(index="census_year", columns=["vintage", "type"])

here, we can clearly see the issue, see the bfill() data that would be pulled, e.g., apartments (1921: 17565.0) and mobile (1961: 1296). This makes no sense. The process should be done census by census. this seems intuitively better, as the best 'guess' for a unknown census would be the 'next' census.

In [ ]:
# check totals
initial_totals = df[df['type']=='total'].pivot(index="census_year", columns=["vintage", "type"])
# FIXME (REPRENDRE) it could be that the interpolation (followed by fix marginals) is wrong due to not being linear/based on year, but based on *decade*
initial_totals

In [ ]:
from uncertimety.interpolate import reconcile_data_with_marginals


# Do it year by year, going backwards in time

# groups = df.groupby('census_year')
# for name, group in groups:
#     test=group.pivot(index=['census_year','vintage'],columns='type',values='dwellings')
#     display(test.head(2))
df3.index = df3.index.astype("int")
census_years =  df3.index.unique().to_list()

df3 = df.set_index(["census_year"]).sort_index(level=0, ascending=False).copy()
new_df = {}
for year in census_years:  # [:5]
    # Take individual census_year
    tsplit3 = df3.loc[year, :].pivot(
        index="vintage", columns="type", values="dwellings"
    )
    
    # Check if there are rows with nans to be bfilled()
    nan_rows = len(tsplit3[tsplit3.isna().any(axis=1)])
    if nan_rows:
        # if you must interpolate, take the data from new_data;
        # check what interpolation year to use; could be the last appended to new_data, but I think it might be better to check the actual year
        next_census = census_years[census_years.index(year) - 1]
        print(f"Census year '{year}': must interpolate from census '{next_census}'")
        # display(tsplit3)
       
        # retrieve dataframe of next census year; overwrite nans with values from the next census, keeping all values of the initial dataframe 
        tsplit3 = tsplit3.combine_first(new_df[next_census])
        # display(tsplit3)

        # NOTE this works decently well, however creates an issue when there are gaps of several Nans between censuses. only backfilling doesn't account for the two likely 'periods', i.e. growth (within span of vintage) followed by stability/decline (after cohort end)
        # FIXME a fix would need to split groups into two: for Nans (cs years) that fall within a cohort, interpolate linearly; for nans that fall after a cohort, interpolate backfill
    else:
        print(f"Census year '{year}': no need for interpolation")
    # TODO there won't ever be interpolation for the most recent year (here, 2021); treat explicitly as specific case? Or we could interpolate forward if there's no available bfill()

    # After interpolation, fix marginals
    new_data, diff = reconcile_data_with_marginals(tsplit3)
    # new_data.stack().reset_index()  # FIXME to add year back?
    # new_data['census_year'] = year  # FIXME to add year back?
    new_df[year] = new_data
    # display(diff)  # can be useful for documentation; maybe check max/min value to see largest diff

    if nan_rows:
        # TODO log this in a notebook or html? see docs/nice_to_have.md
        print(f'result for year {year}:')
        # display(new_data.head(2))
        # display(diff)

res = pd.concat(new_df).stack('type').reset_index().rename(columns={"level_0":"census_year", 0:'dwellings'})
display(res)

In [ ]:
initial_totals

In [ ]:
# TODO Quickcheck (REPRENDRE): plot totals before and after interpolation
fig, axs = plt.subplots(figsize=(12,8))
# axs=axs.ravel()

initial_totals.plot(style='*', ax=axs)
res[res['type']=='total'].pivot(index='census_year', columns='vintage', values='dwellings').plot(style='-', ax=axs)

the result (line) seems to fit pretty close to the data points; however, one issue seems to be 'start' year for some interpolations, e.g., 180k+ dwellings form cohort 1921-1945 in.. 1921 - that's waaay too high.

Mostly, 1921-1945 and 1946-1960 seem at fault - for the other vintages, the first year of each vintage looks good (~10k). 1608-1920 also present 'squiggly' lines. No doubt this is due to long lines of Nans in the initial data.

FIXME REPRENDRE
- maybe let it as is, then smooth the data?
- maybe attempt to fix initial problem, especially the high early counts (e.g., 1921-1945)

In [ ]:
# Test for typesplit based on future interpolation
# Group by census_year and type and sum the dwellings
s = res[res['vintage'] == '1608-2025'].groupby(['census_year', 'type'])['dwellings'].sum()

# For each census_year, divide by the total value
shares = (s / s.xs('total', level='type')).reset_index()
print(shares)

# NOTE Alternate method
# df_pivot = res[res['vintage'] == '1608-2025'].pivot_table(
#     index='census_year',
#     columns='type',
#     values='dwellings',
#     aggfunc='sum'
# )
# shares_df = df_pivot.div(df_pivot['total'], axis=0)
# print(shares_df)

# This shows the typesplit after interpolation
fig, axs = plt.subplots(1,2, figsize=(10,5), sharex=True, sharey=True)
axs=axs.ravel()

sns.lineplot(
    data=shares,
    x="census_year",
    y="dwellings",
    hue="type",
    ax=axs[0],
    # kind="line",
)

sns.lineplot(
    data=tsplit[
        tsplit["type"].isin(
            ["total", "single_detached", "single_attached", "apartments", "mobile"]
        )
    ],
    x="census_year",
    y="dwellings",
    hue="type",
    ax=axs[1],

    # kind="line",
)

for ax in axs:
    ax.set_xlim(1900, 2000)


NOTE: This seems pretty decent. However, there remains the question of what to do with single attached / apartments before 1921 - was it a 'good' year to learn from (i.e., to propagate backwards using bfill)?

In [ ]:
# Attempt full plot by vintage
dfx = res.groupby(['census_year', 'type','vintage']).sum()  # full values per year, census_year and type

dfx['share'] = dfx.groupby(level=[0, 2])["dwellings"].transform(
    lambda x: x / x.loc[x.index.get_level_values(1) == "total"].iloc[0]
)
dfx = dfx.reset_index()
dfx

In [ ]:
agg_types = [
    "total",
    "single_detached",
    "single_attached",
    "apartments",
    "mobile",
    # "other_dwelling",
]

n_vintages = len(dfx["vintage"].unique())
fig, ax = plt.subplots(
    (n_vintages // 2) + 1,
    2,
    figsize=(12, 3 * ((n_vintages // 2) + 1)),
    sharex=True,
    sharey=False,
)
ax = ax.ravel()

for i, vintage in enumerate(dfx["vintage"].unique()):
    tempdata = dfx[
        (dfx["vintage"] == vintage) & (dfx["type"].isin(agg_types))
    ].pivot(index="census_year", columns="type", values="dwellings")
    # display(tempdata)
    tempdata.plot(ax=ax[i])
    ax[i].set_title(vintage)

In [ ]:
# FIXME REPRENDRE - Seems wrong that the 'total' go up and down so much? Why? it could be because the later years are not necessarily 'good', and thus using bfill would 'contradict' earlier counts. However, it more likely is that 'older' counts were not so good, and backfilling introduces contradictions?
# Or it could be an artifact of the IPFN method?
# Mybe try to fix by looking at previous AND following year, and use min? OR, use some kind of smoothing?

In [ ]:
fig, ax = plt.subplots(
    (n_vintages // 2) + 1,
    2,
    figsize=(12, 3 * ((n_vintages // 2) + 1)),
    sharex=True,
    sharey=False,
)
ax = ax.ravel()

for i, vintage in enumerate(dfx["vintage"].unique()):
    dfx[
        (dfx["vintage"] == vintage) & (dfx["type"].isin(agg_types))
    ].pivot(index="census_year", columns="type", values="share").plot(ax=ax[i])
    ax[i].set_title(vintage)

fig.suptitle('typesplit per vintage over time')
plt.tight_layout()

It seems like these should be pretty stable over time? If not, it might imply that there were conversions (?)

##  new attempt for interpolation

In [ ]:
initial_totals

In [ ]:
# Use a combined approach - first backfill years outside cohort; then interpolate within cohort, knowing that there must be zero before cohort start
combined_method = []
groups = df3.reset_index().sort_values('census_year').copy().groupby(by=['vintage','type',])
for name, group in list(groups):
    vintage_start, vintage_end = [int(yr) for yr in name[0].split('-')]
    temp_df = group.copy()

    # First, backfill for stable/declining stocks
    mask = temp_df['census_year'].gt(vintage_end)
    temp_df[mask] = temp_df[mask].bfill()

    # Second, assume linear growth over the census period
    # NOTE: what to do with vintage 1608-1920? for several types, there is basically no data from 1685-1921. It *could* have grown, then shrunk. Here, I'll simply assume a linear growth over the full period, which will then get corrected when applying IPFN. Other methods could also be possible. FIXME: 1608-1920 might need another approach---following the growth curve of 'total' values? this might be naturally fixed by the IPFN later
    threshold = 10
    # mask = temp_df['census_year'].between(vintage_start, vintage_end + threshold, inclusive='both')

    # Do the linear interpolation
    temp = temp_df.copy().set_index('census_year')
    temp = temp.reindex(range(vintage_start - 1, vintage_end + 2))  # include year before and after (e.g., 1607-1921 for 1608-1920)
    temp.loc[vintage_start - 1, 'dwellings'] = 0
    temp['dwellings'] = temp['dwellings'].interpolate(method='index')
    temp['vintage'] = name[0]
    temp['type'] = name[1]

    # Fix indexes
    temp_df = temp_df.set_index(['census_year'])
    temp = temp.reindex(temp_df.index)

    temp_df = temp_df.combine_first(temp).reset_index()
    # display(temp)
    # display(temp_df)

    # retrieve results
    combined_method.append(temp_df)

# THEN, after the interpolation, fix the marginals
combined_method = pd.concat(combined_method)
# (cont'd)

# NOTE on iterpolation. Over these long periods, I know *nothing* of outflows---use ODYM directly? ACTUALLY, this is completely artificial- IF I 'FORCE' stock levels in this way, am I not 'forcing' inflows and 'outflows'? Why am I doing this-- just to compare with the results of ODYM, which relies on population, lifetime, and typesplit? Is this just for comparison? This is mostly a concern for the 1608-1920, which definitely includes outflows. For shorter cohorts,  this might not be such an issue. FIXME REPRENDRE -- maybe need to show the difference between 'real' initial data, 'fixed' (interpolated, ipfn) data, and ODYM results?

In [ ]:
# Fix the marginals and run IPFN
combined_df = {}
for year in census_years:
    print(f" === Census {year} === ")
    # Take individual census_year
    tsplit4 = combined_method.set_index('census_year').loc[year, :].pivot(
        index="vintage", columns="type", values="dwellings"
    )
    # display(tsplit4)
    new_data, diff = reconcile_data_with_marginals(tsplit4)
    # new_data.stack().reset_index()  # FIXME to add year back?
    # new_data['census_year'] = year  # FIXME to add year back?
    combined_df[year] = new_data
    # display(diff)  # can be useful for documentation; maybe check max/min value to see largest diff

c_res = pd.concat(combined_df).stack('type').reset_index().rename(columns={"level_0":"census_year", 0:'dwellings'})
display(c_res)


In [ ]:
n_vintages = len(combined_method["vintage"].unique())
fig, ax = plt.subplots(
    (n_vintages // 2) + 1,
    2,
    figsize=(12, 3 * ((n_vintages // 2) + 1)),
    sharex=True,
    sharey=False,
)
ax = ax.ravel()

for i, vintage in enumerate(combined_method["vintage"].unique()):
    tempdata = combined_method[
        (combined_method["vintage"] == vintage) & (combined_method["type"].isin(agg_types))
    ].pivot(index="census_year", columns="type", values="dwellings")
    # display(tempdata)
    tempdata.plot(ax=ax[i])
    ax[i].set_title(vintage)

In [ ]:
n_vintages = len(c_res["vintage"].unique())
fig, ax = plt.subplots(
    (n_vintages // 2) + 1,
    2,
    figsize=(12, 3 * ((n_vintages // 2) + 1)),
    sharex=True,
    sharey=False,
)
ax = ax.ravel()

for i, vintage in enumerate(c_res["vintage"].unique()):
    tempdata = c_res[
        (c_res["vintage"] == vintage) & (c_res["type"].isin(agg_types))
    ].pivot(index="census_year", columns="type", values="dwellings")
    # display(tempdata)
    tempdata.plot(ax=ax[i])
    ax[i].set_title(vintage)

Here, we can see the IPFN clearly did its job by lowering the amount of dwellings, e.g., in 1608-1920; the results in c_res are also noticeably less 'spiky' than the previous attempt, based on year-by-year interpolation (cf res, in first interpolation attempt). we can also see the spikes suggesting lesser quality data, e.g., in 1608-1920 and 1921-1945




In [ ]:
# fig, axs = plt.subplots(figsize=(12,8))

# # interpolation #2 (linear + bfill)
# c_res[c_res['type']=='total'].pivot(index='census_year', columns='vintage', values='dwellings').plot(ax=axs)

# # interpolation #1 (year-by-year)
# res[res['type']=='total'].pivot(index='census_year', columns='vintage', values='dwellings').plot(style='--', ax=axs)

# # intial values
# initial_totals.droplevel([0,2], axis=1).plot(style='*', ax=axs)

In [ ]:
# Get the sorted list of vintages (assumes all dataframes share the same vintage set)
vintage_order = sorted(c_res[c_res['type']=='total']['vintage'].unique())

# Create a color mapping from vintage to a color.
# Here we use a colormap with enough distinct colors (e.g., tab20)
cmap = plt.get_cmap("tab20")
colors_list = [cmap(i) for i in np.linspace(0, 1, len(vintage_order))]
color_map = dict(zip(vintage_order, colors_list))

# Prepare your pivots ensuring the columns appear in the same order
pivot_c_res = c_res[c_res['type']=='total'].pivot(index='census_year', columns='vintage', values='dwellings').reindex(columns=vintage_order)
pivot_res   = res[res['type']=='total'].pivot(index='census_year', columns='vintage', values='dwellings').reindex(columns=vintage_order)
pivot_initial = initial_totals.droplevel([0,2], axis=1).reindex(columns=vintage_order)

# Build the color list in the same order
colors_used = [color_map[v] for v in vintage_order]

fig, axs = plt.subplots(figsize=(12,8))
# Plot interpolation #2 (linear + bfill)
pivot_c_res.plot(ax=axs, color=colors_used)

# Plot interpolation #1 (year-by-year) with dashed style
pivot_res.plot(ax=axs, style='--', color=colors_used)

# Plot initial values using marker style
pivot_initial.plot(ax=axs, style='*', color=colors_used)

axs.set_xlim(1900, 1960)
axs.set_ylim(0, 1.5e6)

plt.show()
# TODO REPRENDRE analyse/comparaison des résultats ici

looking here, both methods lead to similar results, EXCEPT for 1608-1920 and 1921-1945 (and, to a lesser extent, 1946-1960). It looks as though the method (linear+bfill) is wrong, as it keeps rising *after* 1921 is over (??); it seems to overestimate, while method (year-by-year) seems to underestimate.

method year-by-year also starts *too soon* for 1921-1945, and *too late* for method linear-bfill. FIXME REPRENDRE

In [ ]:
# TODO keep as test for 1946-1960
# dad = {"1851": 0.0, "1861": 0.0, "1871": 0.0, "1881": 0.0, "1891": 0.0, "1901": 0.0, "1911": 0.0, "1921": 0.0, "1931": 0.0, "1941": 0.0, "1951": np.nan, "1956": np.nan, "1961": 517929.0, "1966": 543550.0,}

# dad = pd.DataFrame.from_dict(dad, orient='index')
# dad.index = dad.index.astype('int')
# didx = dad.index
# dad = dad.reindex(list(range(1851,1966+1)))
# dad.loc[:1945] = 0
# dad.interpolate(method='index').loc[didx]

# TODO XXX keep as test for 1921-1945
# dad = {"1851": 0.0, "1861": 0.0, "1871": 0.0, "1881": 0.0, "1891": 0.0, "1901": 0.0, "1911": 0.0, "1921": np.nan, "1931": np.nan, "1941": np.nan, "1951": 301937.0,}

# dad = pd.DataFrame.from_dict(dad, orient='index')
# dad.index = dad.index.astype('int')
# ndad = dad.reindex(list(range(1851, 1951+1)))
# ndad.loc[:1920] = 0
# # dad.index = [int(year) * 12 - (1921*12) for year in dad.index]
# ndad.interpolate(method="index").loc[dad.index]

## Old_cs_data

In [ ]:
# attempt to integrate old_cs_data
from uncertimety.dataprep import load_and_normalize_overwrites

infile = Path(Path.cwd()).resolve().parents[1] / "data" / "old_cs_data.toml"
data = load_and_normalize_overwrites(infile)["census_data"]
groups = pd.DataFrame.from_dict(data).groupby("year")
for name, group in groups:
    display(group["year"].to_numpy()[0])

In [ ]:
infile = Path(Path.cwd()).resolve().parents[1] / "data" / "old_cs_data.toml"
infile

with infile.open("rb") as f:
    data = tomllib.load(f)

data = pd.DataFrame.from_dict(data["census_data"]).replace({-1: np.nan})
data

In [ ]:
# Array tests for round_consistent_sums

c = np.array([1, 2, 3, 4])  #  [10,20,30,40]
desired_sum = 25

# Convert input to a numpy array of floats
arr = np.array(c, dtype=float)
print(arr)

current_sum = arr.sum()
print(current_sum)

# Scaling method 1
# Scale the array to fit the desired sum
if current_sum != desired_sum:
    diff = desired_sum - current_sum
    # overwrite original array so it sums to desired sum
    arr = arr + (arr / current_sum * diff)

print(arr, arr.sum())
# -> confirms same result; proof via paper

In [ ]:
# Array tests for round_consistent_sums

c = np.array([1, 2, 3, 4])  #  [10,20,30,40]
desired_sum = 25

# Convert input to a numpy array of floats
arr = np.array(c, dtype=float)
print(arr)

current_sum = arr.sum()
print(current_sum)

# Scaling method 1
# Scale the array to fit the desired sum
if current_sum != desired_sum:
    arr = arr * (desired_sum / current_sum)

print(arr, arr.sum())

## Random quick tests

In [ ]:
# Go back to tests with initial df
df = pd.DataFrame(
    {
        "vintage": ["1608-2025", "1608-1920", "1921-1945", "1946-1960"],
        "total": [1500, 500, 200, 800],
        "single_detached": [500, 300, 192, 8],
        "other_attached_dwelling": [800, np.nan, 8, 792],
        "other_dwelling": [200, np.nan, np.nan, 0],
    }
)
df

In [ ]:
df.loc[df["total"].idxmax(), "vintage"]

In [ ]:
df.loc[df["vintage"] == "1608-2025"].squeeze()

In [ ]:
def sort_vintage_labels(
    df: pd.DataFrame, vintage_col: str = "vintage", vintage_label: str = "1608-2025"
) -> pd.DataFrame:
    """Sort the dataframe by vintage labels."""
    df = df.sort_values(by=vintage_col)
    idx = df.index
    target_idx = df[df[vintage_col] == vintage_label].index.tolist()
    print(idx, target_idx)
    idx = target_idx + idx.difference(target_idx).tolist()

    return df.loc[idx, :]


sort_vintage_labels(df)

In [ ]:
df[["vintage", "total"]]

In [ ]:
def check_series_sum(
    df: pd.DataFrame,
    target: str,  # either a row or column label
    groupby: str = "vintage",
    total_label: str = None,
    atol: float = 5,
    rtol: float = 1e-5,
    axis: bool = 0,
) -> Tuple[bool, pd.Series]:
    """
    Check if components sum to total within grouped data.

    Args:
        df: DataFrame with data to check
        value_col: Column containing values to sum (e.g., 'total')
        groupby: Column to group by (e.g., 'vintage')
        vintage_total: Label in groupby representing the total
        atol: Absolute tolerance for comparison (used by np.isclose)
        rtol: Relative tolerance for comparison (used by np.isclose)

    Returns:
        Tuple of (bool, Series) where:
            - bool indicates if all groups pass the check
            - Series contains the difference between total and sum of components for each group
    """
    if total_label is None:
        total_label = "total" if axis == 0 else "1608-2025"  # FIXME use constants?

    grouped = df.groupby(groupby).sum()

    if axis == 0:  # for rows, sum over types
        try:
            total = grouped.loc[target, total_label]
            components = grouped.loc[target].drop(total_label)
        except KeyError as err:
            raise KeyError(
                f"Target '{target}' not found in DataFrame {grouped.index}: {err}. Check that target and axis are consistent."
            )
    elif axis == 1:  # for columns, sum over vintages
        try:
            total = grouped.loc[total_label, target]
            components = grouped[target].drop(total_label)
        except KeyError as err:
            raise KeyError(
                f"Target '{target}' not found in DataFrame {grouped.columns}: {err}"
            )
    else:
        raise ValueError(f"Invalid axis {axis}. Use 0 for rows or 1 for columns.")

    # Sum the components
    component_sum = components.fillna(0).sum()

    # Calculate difference
    difference = total - component_sum

    # Check if within tolerance (using numpy's isclose for both absolute and relative tolerance)
    check_passed = np.isclose(total, component_sum, rtol=rtol, atol=atol)

    result = pd.Series(
        {
            "target": target,
            "total_label": total_label,
            "total": total,
            "component_sum": component_sum,
            "difference": difference,
            "check_passed": check_passed,
        }
    )

    return check_passed, result


In [ ]:
display(df)
# display(check_sums(df, 'single_detached', axis=0))  # this (correctly) breaks if target is set to a dwelling type;
display(check_series_sum(df, "single_detached", axis=1))

In [ ]:
display(df)
# display(check_sums(df, '1608-1920', axis=1))  # this (correctly) breaks if target is set to a vintage;
display(check_series_sum(df, "1608-1920", axis=0))

In [ ]:
df.groupby("vintage").sum().columns  # .loc['1608-1920'].drop('total')

In [ ]:
df.groupby("vintage").sum()["total"].drop("1608-2025")

In [ ]:
# for rows, sum over types
grouped = df.groupby("vintage").sum()
grouped
# display(grouped.loc['1608-2025','total'])
# grouped.drop(columns='total').loc['1608-2025']

In [ ]:
# for cols, sum over vintages
grouped = df.groupby("vintage").sum()
display(grouped.loc["1608-2025", "total"])
display(grouped.drop(index="1608-2025")["total"])

In [ ]:
display(df)
check_series_sum(df, "total", axis=1)


In [ ]:
def check_marginals(
    df: pd.DataFrame,
    groupby: str = "vintage",
    vintage_label: str = "1608-2025",
    type_label: str = "total",
    atol: float = 5,
    rtol: float = 1e-5,
) -> Tuple[bool, Dict[str, Any]]:
    """
    Check if the marginals (total counts) are consistent across vintages and types.

    Args:
        df: DataFrame with vintage and dwelling type data
        groupby: Column containing vintage labels (e.g., 'vintage')
        vintage_label: The vintage label representing the total (e.g., "1608-2025")
        type_label: The column name representing total dwellings (e.g., "total")
        atol: Absolute tolerance for comparison
        rtol: Relative tolerance for comparison

    Returns:
        Tuple[bool, Dict]:
            - Boolean indicating if marginals are consistent
            - Dictionary with detailed results including:
                - 'sum_by_type': Series with type sum details
                - 'sum_by_vintage': Series with vintage sum details
                - 'sums_match': Whether component sums match
                - 'difference': Difference between component sums
    """
    # Check that required columns exist
    if type_label not in df.columns or groupby not in df.columns:
        msg = f"Columns {type_label} or {groupby} are missing from DataFrame columns: {df.columns}"
        print(msg)
        raise ValueError(msg)

    # Check that the required vintage label exists in the groupby column
    if vintage_label not in df[groupby].values:
        msg = f"Vintage label '{vintage_label}' not found in '{groupby}' column: {df[groupby].unique()}"
        print(msg)
        raise ValueError(msg)

    try:
        # Check if dwelling types sum to the vintage total
        type_passed, sum_by_type = check_series_sum(
            df, target=vintage_label, groupby=groupby, atol=atol, rtol=rtol, axis=0
        )

        # Check if vintages sum to the type total
        vintage_passed, sum_by_vintage = check_series_sum(
            df, target=type_label, groupby=groupby, atol=atol, rtol=rtol, axis=1
        )
        # There are two things we need to check: first, that the component sums match for types and vintages agree; second, that this matches the total value

        # Compare the component sums from both approaches (should be equal)
        sums_match = np.isclose(
            sum_by_type["component_sum"],
            sum_by_vintage["component_sum"],
            atol=atol,
            rtol=rtol,
        )

        difference = sum_by_type["component_sum"] - sum_by_vintage["component_sum"]

        # Combine all checks
        all_passed = type_passed and vintage_passed and sums_match

        results = {
            "sum_by_type": sum_by_type,
            "sum_by_vintage": sum_by_vintage,
            "sums_match": sums_match,
            "difference": difference,
        }

        if all_passed:
            print(
                f"Marginal check passed: type sum {sum_by_type['component_sum']} "
                f"and vintage sum {sum_by_vintage['component_sum']} agree."
            )
        else:
            if sums_match:
                print(
                    f"Marginal check failed: type sum {sum_by_type['total']} "
                    f"and vintage sum {sum_by_vintage['total']} differ, but "
                    f"the component sums match."
                )
                # TODO return all_passed True here?
            else:
                print(
                    f"Marginal check failed: type_passed={type_passed}, "
                    f"vintage_passed={vintage_passed}, sums_match={sums_match}, "
                    f"difference={difference}"
                )
    except KeyError as err:
        print(f"Error checking marginals: {err}")
        return False, {"error": str(err)}

    return all_passed, results

In [ ]:
check_marginals(df)
# ok this seems to work well

In [ ]:
# now, for the harder cases
df_wrong_total = df.copy()
df_wrong_total.loc[0, "total"] = 1200
check_marginals(df_wrong_total)

In [ ]:
df_components_sum_mismatch = df.copy()
df_components_sum_mismatch.loc[0, "other_attached_dwelling"] = 700
check_marginals(df_components_sum_mismatch)

In [ ]:
# how to apply df-wide check series sum?
display(df)
df.apply(lambda s: check_series_sum(df, s["vintage"]), axis=1)  # matches for all rows

results = df.apply(
    lambda s: check_series_sum(df, s.name, axis=1)[0] if s.name != "vintage" else None,
    axis=0,
    result_type="expand",
)  # matches for all columns. # FIXME super hard to read, retrofit implementation of check_series_sum or

results
# [(passed, details) for passed, details in [a if a is not None else (None, None) for a in results]]
# check_marginals(df)


In [ ]:
pd.Series(data={"a": 1, "b": 2})

In [ ]:
df

In [ ]:
# sorting issues
import random

# 'quicksort' works
# 'mergesort'
# 'heapsort'
vintages = [
    "1608-2025",
    "1608-1920",
    "1921-1945",
    "1946-1960",
    "1961-1970",
    "1971-1980",
    "1981-1990",
    "1991-1995",
    "1996-2000",
    "2001-2005",
    "2006-2010",
    "2011-2015",
    "2016-2020",
    "2021-2025",
]
random.shuffle(vintages)
sdf = pd.DataFrame(vintages, columns=["vintage"])
# sdf.sort_values('vintages', kind='heapsort') #


def sort_vintage_labels(
    df: pd.DataFrame, vintage_col: str = "vintage", vintage_label: str = "1608-2025"
) -> pd.DataFrame:
    """Sort the dataframe by vintage labels."""
    df = df.sort_values(by=vintage_col, kind="quicksort")
    idx = df.index
    target_idx = df[df[vintage_col] == vintage_label].index.tolist()
    idx = target_idx + idx.difference(target_idx).tolist()
    df = df.loc[idx, :].reset_index(drop=True)  # Reset the index here
    return df


sort_vintage_labels(sdf)  # FIXME sorting alphabetically does not work

In [ ]:
df.loc[((df.notna().any(axis=1)) & (df["vintage"] != "1608-2025"))]


def _find_compatible_vintages(df, vintage: str, sep="-"):
    # Find non-empty rows
    non_nans = df.loc[(df.notna().any(axis=1))]

    # Find compatible vintages
    start, end = [int(years) for years in vintage.strip().split(sep)]

    compatible = df.loc[
        (df["vintage"].apply(lambda x: int(x.split("-")[0])) >= start)
        & (df["vintage"].apply(lambda x: int(x.split("-")[1])) <= end)
    ]

    # Get the intersection
    target_indices = non_nans.index.intersection(compatible.index)

    return df.loc[target_indices]


display(df)
a = _find_compatible_vintages(df, "1920-1970")

# df.loc[((df.isna().any(axis=1)) & _get_compatible_vintages(df, '1920-1970').index)]
display(a)
a.drop(columns=["vintage"]).sum(min_count=2).to_dict()

In [ ]:
new_df = pd.DataFrame().reindex_like(df).drop([1, 2, 3]).astype({"vintage": str})
new_df.loc[0, :] = ["1961-1970"] + [np.nan] * (len(df.columns) - 1)
new_df = pd.concat([df, new_df], ignore_index=True)

nan_rows = new_df.drop("vintage", axis=1).isna().all(axis=1)
new_df.loc[nan_rows, "vintage"].to_list()
# new_df.iloc[:,1:]
new_df[["total", "single_detached", "other_dwelling"]].sum()

In [ ]:
compatible = _find_compatible_vintages(new_df, "1946-1970")
numeric_cols = [
    "total",
    "single_detached",
    "other_attached_dwelling",
]
compatible[numeric_cols].sum(min_count=1)

bb = new_df.copy()
bb.loc[bb["vintage"] == "1961-1970", numeric_cols] = compatible[numeric_cols].sum(
    min_count=1
)

bb

In [ ]:
a = df.copy().set_index("vintage")
display(a)
target_types = ["single_detached", "other_dwelling"]
expected_total = a.loc["1608-2025", "total"]
expected_sum_over_types = a.drop("1608-2025").loc[:, target_types].sum(axis=1)
expected_sum_over_vintages = a.drop("total", axis=1).loc["1608-2025"]

expected_total, expected_sum_over_types, expected_sum_over_vintages
expected_sum_over_types

In [ ]:
df.set_index("vintage").index.to_list()

In [ ]:
df.set_index("vintage").isna().sum(axis=1)
df.set_index("vintage").isna().sum(axis=0)
# df.set_index('vintage').isna().sum().sum()


# [(df['vintage'][vintage_index], df.columns.to_list()[type_index]) for vintage_index, type_index in np.argwhere(np.isnan(df.set_index('vintage')))]
[
    (
        df.set_index("vintage").index.to_list()[vintage_index],
        df.columns.to_list()[type_index],
    )
    for vintage_index, type_index in np.argwhere(np.isnan(df.set_index("vintage")))
]

df.set_index("vintage").isna().sum().sum()


In [ ]:
display(df)


def test_fun(
    df,
    groupby="vintage",
    vintage_label="1608-2025",
    type_label="total",
    atol=5,
    rtol=1e-5,
):
    working_df = df.copy()
    working_df = working_df.set_index(groupby)

    expected_total = working_df.loc[vintage_label, type_label]
    expected_sum_over_types = working_df.drop(vintage_label).loc[:, type_label]
    expected_sum_over_vintages = working_df.drop(type_label, axis=1).loc[vintage_label]

    working_df = working_df.drop(vintage_label).drop(type_label, axis=1)

    marginal_totals = pd.Series(
        data={
            "expected_sum_over_types": expected_sum_over_types.sum(),
            "expected_sum_over_vintages": expected_sum_over_vintages.sum(),
        }
    )  # actual marginals from the DF, not the calculated ones

    sum_over_types = working_df.sum(axis=1)  # should be equal to 'total'
    sum_over_vintages = working_df.sum(axis=0)  # should be equal to '1608-2025'

    diff_types = sum_over_types - expected_sum_over_types
    diff_vintages = sum_over_vintages - expected_sum_over_vintages

    results = {
        "sum_over_types": (
            diff_types,
            np.isclose(expected_sum_over_types, sum_over_types, atol=atol, rtol=rtol),
        ),
        "sum_over_vintages": (
            diff_vintages,
            np.isclose(
                expected_sum_over_vintages, sum_over_vintages, atol=atol, rtol=rtol
            ),
        ),
        "marginals_agree": (
            marginal_totals,
            np.isclose(
                pd.Series([expected_total] * len(marginal_totals)),
                marginal_totals,
                atol=atol,
                rtol=rtol,
            ),
        ),
    }

    return results


results = test_fun(df)
results

In [ ]:
#
df.columns.name = "type"
df.T.groupby("type").sum()

In [ ]:
# TODO test fpr check_marginals


class TestCheckMarginals:
    def test_exact_match(self, sample_df):
        """Test when row sums exactly match column sums."""
        passed, details = check_marginals(sample_df, "vintage", "total", "1608-2025")
        assert passed
        assert details["sum_by_column"] == 1000
        assert details["sum_by_row"] == 1000
        assert details["difference"] == 0

    def test_with_nans(self):
        """Test with NaN values that should be treated as zeros."""
        df = pd.DataFrame(
            {
                "vintage": ["1608-2025", "1608-1920", "1921-1945"],
                "total": [1000, 450, 550],
                "other_attached_dwelling": [800, np.nan, 400],
                "other_dwelling": [200, 50, 150],
            }
        )

        passed, details = check_marginals(df, "vintage", "total", "1608-2025")
        assert passed
        assert details["sum_by_column"] == 1000
        assert details["sum_by_row"] == 1000

    def test_with_tolerance(self, sample_df):
        """Test with values within tolerance."""
        df = sample_df.copy()
        df.loc[df["vintage"] == "1608-2025", "other_attached_dwelling"] = 801

        # Should fail with tight tolerance
        passed, details = check_marginals(df, "vintage", "total", "1608-2025", atol=0.5)
        assert not passed
        assert details["sum_by_row"] == 1001

        # Should pass with looser tolerance
        passed, details = check_marginals(df, "vintage", "total", "1608-2025", atol=2)
        assert passed

In [ ]:
# Check the desired dataset output
df = pd.read_csv(
    "/home/cbreton/dev/cbreton026/uncertimety/data/clean/curated_census_dwelling_stock.csv"
)

tidy = (
    df.set_index(["census_year", "vintage"])
    .stack(level=0, future_stack=True)
    .reset_index(name="dwellings")
    .rename(columns={"level_2": "type"})
    .astype({"census_year": int, "vintage": str, "type": str, "dwellings": float})
)
display(tidy)
tidy["type"].unique()

In [ ]:
# prepare interpolation
infile = "/home/cbreton/dev/cbreton026/uncertimety/data/clean/fulldata.parquet"
df = pd.read_parquet(infile)
df

In [ ]:
# Fill missing values using interpolation
groups = df.groupby(["type", "vintage"])
group_list = list(groups)  # convert groupby iterator to list

new_groups = {}

for name, group in group_list:
    new_df = group.copy()
    new_df["dwellings"] = new_df["dwellings"].bfill()
    # display(new_df)  # FIXME TEMP REMOVE

    new_groups[name] = new_df

concat_df = pd.concat(new_groups.values())  # .set_index(['vintage','type'])
display(concat_df.head())


In [ ]:
agg_types = [
    "total",
    "single_detached",
    "single_attached",
    "apartments",
    "mobile",
    # "other_dwelling",
]

n_vintages = len(df["vintage"].unique())
fig, ax = plt.subplots(
    (n_vintages // 2) + 1,
    2,
    figsize=(12, 3 * ((n_vintages // 2) + 1)),
    sharex=True,
    sharey=False,
)
ax = ax.ravel()

for i, vintage in enumerate(df["vintage"].unique()):
    concat_df[
        (concat_df["vintage"] == vintage) & (concat_df["type"].isin(agg_types))
    ].pivot(index="census_year", columns="type", values="dwellings").plot(ax=ax[i])

In [ ]:
# attempt masks to keep initial values
display(concat_df)

df[df.isna().any(axis=1)]
# concat_df.where(df.isna().any(axis=1))
mask_na = (
    df.groupby(["census_year", "vintage", "type"]).sum(min_count=1).reset_index().isna()
)  # .any(axis=1)
mask_raw = (
    df.groupby(["census_year", "vintage", "type"])
    .sum(min_count=1)
    .reset_index()
    .notna()
)
concat_df.groupby(["census_year", "vintage", "type"]).sum(
    min_count=1
).reset_index().where(mask_raw)

In [ ]:
# test with assertframeequal? make unittests to ensure the index is accurate?
display(concat_df.where(mask_raw).tail(20))
display(
    concat_df.groupby(["census_year", "vintage", "type"])
    .sum(min_count=1)
    .reset_index()
    .where(mask_raw)
    .tail(20)
)

"""To ensure the mask applies to exactly the same columns in both the original and the interpolated DataFrame, you should:

Generate the mask using a fixed, explicit list of columns (the “by” columns plus the numeric columns you want to check).
Reindex or select those exact columns from the interpolated DataFrame before applying the mask.
Optionally, store the mask’s column order and verify that the interpolated DataFrame has the same column order.
For example, you could do the following:"""
# # Generate the mask on the original dataset
# by = ["census_year", "vintage", "type"]
# grouped = original_df.groupby(by).sum(min_count=1).reset_index()
# mask = grouped.notna()

# # When processing the interpolated DataFrame, ensure the same aggregation and ordering is used:
# interp_grouped = interpolated_df.groupby(by).sum(min_count=1).reset_index()

# # Reindex the columns to exactly match the mask, if needed:
# interp_grouped = interp_grouped.reindex(columns=mask.columns)

# # Now apply the mask (or use it to verify consistency)
# result = interp_grouped.where(mask)

In [ ]:
data = {
    "vintage": ["1608-2025", "1608-1920", "1921-1945", "1946-1960"],
    "total": [150, 260, 300, 140],
    "single_attached": [10, 20, 50, 30],
    "single_detached": [50, 100, 120, 60],
    "apartments": [60, 80, 100, 40],
    "mobile": [40, 80, 80, 40],
}
df = pd.DataFrame(data)
df

# df.iloc[1:, 2:] for sums
# 280, 350, 170 -> 800 (sum over columns, i.e., sum by cohort)
# 100, 280, 220, 200 -> 800 (sum over rows, i.e., sum by type)


In [ ]:
df.set_index("vintage").iloc[1, 1:].sum()